# Pipeline Dataset: Pengumpulan, Konstruksi, dan StatistikNotebook ini membangun dataset verifikasi gambar-teks dari portal berita, mulai daripengambilan URL sampai tabel statistik. Satu berkas, dijalankan berurutan.**Tahap**1. Parameter dan utilitas2. Pengumpulan URL3. Parsing artikel4. Penyaringan dan deduplikasi5. Konstruksi pasangan `true` dan `out-of-context`6. Konstruksi `miscaptioned` dengan penulisan ulang model bahasa7. Variasi kedua per artikel8. Penyusunan dan pembagian data9. Statistik dan sidik jari**Keluaran** di `data/`:| Berkas | Isi ||---|---|| `mc_raw.jsonl` | respons mentah model, termasuk yang ditolak dan alasannya || `pairs.jsonl` | dataset final dengan kolom `fold` |

## 0. ParameterKunci API dibaca dari berkas `.env` di root repo, atau dari variabel lingkungan`OPENCODE_API_KEY`. Berkas `.env` sudah masuk `.gitignore`.Respons model disimpan bertahap di `data/mc_raw.jsonl`. Artikel yang sudah punyarespons tidak dipanggil ulang, sehingga menjalankan notebook kembali hanya menyusunulang hasil yang ada.

In [ ]:
from pathlib import Pathimport difflib, hashlib, html as ihtml, json, os, random, re, timeimport urllib.error, urllib.request, uuidfrom collections import Counter, defaultdictfrom concurrent.futures import ThreadPoolExecutortry:    from tqdm.auto import tqdmexcept ImportError:    class tqdm:        def __init__(self, iterable=None, total=None, desc=None, unit=None, disable=False):            self.iterable = iterable if iterable is not None else []        def __iter__(self):            return iter(self.iterable)        def update(self, n=1):            pass        def set_postfix(self, **kw):            pass        def close(self):            passdef repo_root(start: Path | None = None) -> Path:    p = (start or Path.cwd()).resolve()    while p != p.parent:        if (p / ".git").exists():            return p        p = p.parent    return (start or Path.cwd()).resolve()ROOT = repo_root()OUT = ROOT / "data"OUT.mkdir(parents=True, exist_ok=True)def load_env(path: Path):    if not path.exists():        return    for line in path.read_text().splitlines():        if "=" in line and not line.strip().startswith("#"):            k, v = line.split("=", 1)            os.environ.setdefault(k.strip(), v.strip())load_env(ROOT / ".env")LIMIT = 3900WORKERS = 8SEED = 42VERBOSE = TrueRECOLLECT = FalseMODEL = "deepseek-v4.1-flash"API_URL = "https://opencode.ai/inference/go/openai/v1/chat/completions"BATCH = 8UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120 Safari/537.36"}rng_ooc = random.Random(SEED + 1)def write_json(path: Path, obj):    tmp = path.with_suffix(path.suffix + ".tmp")    tmp.write_text(json.dumps(obj, indent=0, ensure_ascii=False))    os.replace(tmp, path)def write_jsonl(path: Path, rows):    tmp = path.with_suffix(path.suffix + ".tmp")    with open(tmp, "w") as f:        for r in rows:            f.write(json.dumps(r, ensure_ascii=False) + "\n")    os.replace(tmp, path)def read_jsonl(path: Path):    if not path.exists():        return []    return [json.loads(l) for l in open(path) if l.strip()]def sha256_of(path: Path) -> str:    h = hashlib.sha256()    with open(path, "rb") as f:        for chunk in iter(lambda: f.read(1 << 20), b""):            h.update(chunk)    return h.hexdigest()print("root:", ROOT)print("keluaran:", OUT)print("limit:", LIMIT, "| model:", MODEL)

## 1. Utilitas HTTP

In [ ]:
def get(url: str, retries: int = 2, timeout: int = 30) -> str:    for i in range(retries):        try:            req = urllib.request.Request(url, headers=UA)            with urllib.request.urlopen(req, timeout=timeout) as r:                return r.read().decode("utf8", "ignore")        except Exception:            if i == retries - 1:                raise            time.sleep(1)print(get("https://www.kompas.com/tag/banjir?page=1")[:80])

## 2. Pengumpulan URLHasil disimpan ke `data/urls.json` dan dipakai ulang pada run berikutnya.

In [ ]:
ARTICLE = re.compile(r'href="(https://[a-z]+\.kompas\.com/read/\d{4}/\d{2}/\d{2}/[^"]+)"')def add_urls(urls: list[str], seen: set[str], url: str) -> int:    try:        page = get(url)    except Exception:        return 0    added = 0    for u in ARTICLE.findall(page):        if u not in seen:            seen.add(u)            urls.append(u)            added += 1    return addedFRONTS = [    "https://regional.kompas.com/", "https://nasional.kompas.com/",    "https://megapolitan.kompas.com/", "https://tekno.kompas.com/",    "https://money.kompas.com/", "https://bola.kompas.com/",    "https://lifestyle.kompas.com/", "https://health.kompas.com/",    "https://edukasi.kompas.com/", "https://travel.kompas.com/",    "https://properti.kompas.com/", "https://otomotif.kompas.com/",    "https://sains.kompas.com/", "https://internasional.kompas.com/",]TAGS = [    "banjir", "kebakaran", "kecelakaan", "longsor", "gempa", "kriminal", "hukum",    "demo", "kesehatan", "pendidikan", "ekonomi", "politik", "pemilu", "olahraga",    "teknologi", "otomotif", "travel", "gaya-hidup", "properti", "internasional",    "jakarta", "surabaya", "bandung", "semarang", "medan", "makassar", "palembang",    "yogyakarta", "malang", "denpasar", "cuaca", "lingkungan", "energi", "pangan",    "transportasi", "infrastruktur", "kesehatan-masyarakat", "viral", "hoaks",    "bencana", "banjir-bandang", "tanah-longsor", "gempa-bumi", "kebakaran-hutan",    "kecelakaan-lalu-lintas", "kriminalitas", "korupsi", "pengadilan", "pemerintahan",    "kesejahteraan", "lingkungan-hidup", "pertanian", "perikanan", "perdagangan",]def collect(limit: int) -> list[str]:    urls, seen = [], set()    pages = [(f, 0) for f in FRONTS]    pages += [(tag, page) for tag in TAGS for page in range(1, 21)]    bar = tqdm(pages, desc="URL", unit="halaman", disable=not VERBOSE)    for name, page in bar:        link = name if page == 0 else f"https://www.kompas.com/tag/{name}?page={page}"        add_urls(urls, seen, link)        bar.set_postfix(url=len(urls))        if len(urls) >= limit:            break    bar.close()    return urls[:limit]

In [ ]:
URLS = OUT / "urls.json"if URLS.exists() and not RECOLLECT:    urls = json.loads(URLS.read_text())[:LIMIT]    print(f"memakai daftar tersimpan: {len(urls)} URL")else:    t0 = time.time()    urls = collect(LIMIT)    write_json(URLS, urls)    print(f"URL terkumpul: {len(urls)} | {time.time()-t0:.0f} detik")print(urls[0])

## 3. Parsing artikelEmpat field diambil: judul, gambar utama, caption, dan kanal. `og:image` harus samadengan gambar yang diberi caption, karena hanya pasangan yang sama yang layak menjadikelas `true`.Hasil ditambahkan ke `data/articles_raw.jsonl`; halaman yang sudah ada tidak diambil ulang.

In [ ]:
def plain(s: str) -> str:    return re.sub(r"\s+", " ", ihtml.unescape(re.sub(r"<[^>]+>", " ", s))).strip()def parse(url: str) -> dict:    page = get(url)    title = re.search(r"<h1[^>]*>(.*?)</h1>", page, re.S)    og = re.search(r'property="og:image" content="([^"]+)"', page)    cap = re.search(r'<div class="photo__caption[^"]*"[^>]*>(.*?)</div>', page, re.S)    cat = re.search(r"//([a-z]+)\.kompas\.com/", url)    return {        "url": url,        "title": plain(title.group(1)) if title else "",        "image": og.group(1) if og else "",        "caption": plain(cap.group(1)) if cap else "",        "category": cat.group(1) if cat else "",    }def safe(u: str):    try:        return parse(u)    except Exception:        return None

In [ ]:
RAW = OUT / "articles_raw.jsonl"done = {a["url"] for a in read_jsonl(RAW)}todo = [u for u in urls if u not in done]print(f"sudah ada {len(done)} | sisa {len(todo)}")t0 = time.time()if not todo:    print("tidak ada halaman baru")else:    with open(RAW, "a") as f, ThreadPoolExecutor(max_workers=WORKERS) as ex:        it = tqdm(ex.map(safe, todo), total=len(todo), desc="parsing",                  unit="artikel", disable=not VERBOSE)        for a in it:            if a is not None:                f.write(json.dumps(a, ensure_ascii=False) + "\n")    print(f"selesai parsing | {time.time()-t0:.0f} detik")

## 4. Penyaringan dan deduplikasiEnam kriteria berurutan. Duplikat gambar dideteksi setelah parameter `filters:` dibuang.Pembacaan raw memakai dedup per URL.

In [ ]:
SKIP_IMAGE = re.compile(r"(logo|default|placeholder|avatar|icon)", re.I)IMG_KEY = re.compile(r"/filters:[^/]+")raw, seen_url = [], set()for a in tqdm(read_jsonl(RAW), desc="filter", unit="artikel", disable=not VERBOSE):    if a["url"] in seen_url:        continue    seen_url.add(a["url"])    raw.append(a)arts, reject = [], Counter()for a in raw:    if not a["image"]:        reject["tanpa gambar"] += 1    elif SKIP_IMAGE.search(a["image"]):        reject["gambar logo atau placeholder"] += 1    elif len(a["caption"].split()) < 6:        reject["caption terlalu pendek atau kosong"] += 1    elif len(a["title"].split()) < 4:        reject["judul terlalu pendek"] += 1    else:        arts.append(a)seen, uniq = set(), []for a in arts:    key = IMG_KEY.sub("", a["image"])    if key in seen:        reject["gambar duplikat"] += 1        continue    seen.add(key)    uniq.append(a)print(f"lolos filter: {len(uniq)}")for k, v in reject.most_common():    print(f"  {k}: {v}")write_jsonl(OUT / "articles.jsonl", uniq)

## 5. Pasangan `true` dan `out-of-context``out-of-context` memasangkan caption artikel A dengan gambar artikel B dari kanal lain.Pasangan finalnya dibentuk ulang di dalam fold pada tahap pembagian data; di sini hanyadibuat kerangka dengan jumlah yang sama.

In [ ]:
pairs = [{"label": "true", "image": a["image"], "caption": a["caption"],          "source": a["url"], "category": a["category"]} for a in uniq]by_cat = defaultdict(list)for a in uniq:    by_cat[a["category"]].append(a)for a in tqdm(uniq, desc="out-of-context", unit="artikel", disable=not VERBOSE):    pool = [b for b in by_cat[a["category"]]            if b["url"] != a["url"] and b["image"] != a["image"]]    if not pool:        pool = [b for b in uniq if b["url"] != a["url"] and b["image"] != a["image"]]    if pool:        b = rng_ooc.choice(pool)        pairs.append({"label": "out-of-context", "image": b["image"],                      "caption": a["caption"], "source": a["url"],                      "category": a["category"]})print("true:", sum(1 for p in pairs if p["label"] == "true"),      "| out-of-context:", sum(1 for p in pairs if p["label"] == "out-of-context"))

## 6. Konstruksi `miscaptioned`Dua peran dipisah:1. **Aturan** memilih satu entitas yang terlihat pada gambar, dan melewati kata yang   diikuti nama diri atau membentuk frasa khusus.2. **Model bahasa** menulis ulang kalimat dengan entitas lain sejenis, tanpa mengubah   nama, tempat, tanggal, angka, dan teks dalam tanda kurung.Setiap keluaran divalidasi dengan enam kriteria. Yang lolos disimpan, yang gagal dicatatbeserta alasannya.

In [ ]:
TERLIHAT = {    "bencana": ["banjir", "kebakaran", "tanah longsor", "longsor", "gempa bumi",                "gempa", "kekeringan", "erupsi", "angin puting beliung"],    "kendaraan": ["sepeda motor", "motor", "mobil", "truk", "bus", "kereta api",                  "kapal", "sepeda", "pesawat"],    "bangunan": ["rumah", "sekolah", "pasar", "kantor", "jembatan", "masjid", "gereja",                 "gudang", "toko", "terminal", "stasiun", "bandara", "pelabuhan",                 "gedung", "jalan", "lapangan", "taman", "tenda", "posko", "dermaga"],    "orang": ["polisi", "tentara", "petugas", "relawan", "warga", "siswa", "guru",              "pedagang", "nelayan", "petani", "petugas pemadam kebakaran"],    "alam": ["sawah", "ladang", "hutan", "pantai", "sungai", "gunung", "danau", "perahu"],}def find_entity(caption: str):    for kat, words in TERLIHAT.items():        for w in words:            m = re.search(rf"\b{re.escape(w)}\b", caption, flags=re.I)            if m and not re.match(r"\s+[A-Z]", caption[m.end():m.end() + 2]):                return w, kat    return None, NoneSYS = (    "Kamu editor caption berita berbahasa Indonesia. Pada setiap baris ada entitas di "    "dalam tanda kurung siku beserta jenisnya. Ganti entitas itu dengan entitas lain "    "dari jenis yang sama, sehingga caption tidak lagi cocok dengan fotonya, tetapi "    "kalimatnya tetap wajar dan panjangnya mirip. Pilih pengganti yang spesifik dan "    "bervariasi. Dilarang mengubah nama orang, nama tempat, nama lembaga, tanggal, "    "angka, dan teks dalam tanda kurung. Dilarang menambah atau menghapus informasi lain. "    "Balas HANYA dengan JSON: {\"items\":[{\"i\":0,\"caption\":\"...\"}]} tanpa penjelasan.")NUM = re.compile(r"\d")def name_positions(tokens):    marks = [bool(w[:1].isupper()) for w in tokens]    out, i = set(), 0    while i < len(tokens):        if marks[i]:            j = i            while j + 1 < len(tokens) and marks[j + 1]:                j += 1            if j > i:                out.update(range(i, j + 1))            i = j + 1        else:            i += 1    return outdef validate(orig: str, new: str):    if not new or len(new) < 20:        return False, "kosong atau terlalu pendek", []    a, b = orig.split(), new.split()    if not (0.85 <= len(b) / max(1, len(a)) <= 1.25):        return False, "panjang menyimpang", []    sm = difflib.SequenceMatcher(None, a, b)    names = name_positions(a)    changed = []    for tag, i1, i2, j1, j2 in sm.get_opcodes():        if tag == "equal":            continue        old, rep = a[i1:i2], b[j1:j2]        if any(NUM.search(w) for w in old + rep):            return False, "mengubah angka", old + rep        if any(w.startswith("(") or w.endswith(")") for w in old + rep):            return False, "mengubah teks kurung", old + rep        if any(k in names for k in range(i1, i2)):            return False, "mengubah nama diri", old + rep        changed += old + rep    if not changed:        return False, "tidak ada perubahan", []    if len(changed) > 4:        return False, "terlalu banyak perubahan", changed    return True, "ok", changed

### 6.1 Pemanggilan modelSatu permintaan memuat delapan caption. Hasilnya ditulis ke `data/mc_raw.jsonl` segerasetelah diterima, sehingga proses yang terputus bisa dilanjutkan.

In [ ]:
def api_key() -> str:    key = os.environ.get("OPENCODE_API_KEY", "")    if not key:        raise RuntimeError(            "OPENCODE_API_KEY belum diisi. Letakkan di berkas .env pada root repo, "            "atau set sebagai variabel lingkungan.")    return keydef call_llm(messages, max_tokens=1200, temp=0.2, retries=4):    key = api_key()    body = {"model": MODEL, "messages": messages, "temperature": temp,            "max_tokens": max_tokens, "reasoning_effort": "none"}    last = None    for k in range(retries):        try:            req = urllib.request.Request(                API_URL, data=json.dumps(body).encode(),                headers={"Authorization": f"Bearer {key}",                         "Content-Type": "application/json",                         "User-Agent": "image-text-verification/1.0",                         "x-opencode-session": str(uuid.uuid4())})            r = json.load(urllib.request.urlopen(req, timeout=180))            return r["choices"][0]["message"].get("content") or "", r.get("usage", {})        except urllib.error.HTTPError as e:            if e.code in (429, 500, 502, 503, 504):                time.sleep(2 ** k)                last = f"HTTP {e.code}"                continue            raise RuntimeError(f"HTTP {e.code}: {e.read()[:200].decode(errors='ignore')}")        except Exception as e:            last = str(e)            time.sleep(2 ** k)    raise RuntimeError(f"gagal setelah {retries} percobaan: {last}")def parse_json(text: str) -> dict:    t = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()    try:        d = json.loads(t)    except Exception:        m = re.search(r"\{.*\}", t, flags=re.S)        if not m:            return {}        d = json.loads(m.group(0))    items = d.get("items", d if isinstance(d, list) else [])    return {int(it["i"]): str(it["caption"]).strip()            for it in items if isinstance(it, dict) and "i" in it and "caption" in it}

In [ ]:
MC_RAW = OUT / "mc_raw.jsonl"seen_src = {r["source"] for r in read_jsonl(MC_RAW)}cand = [a for a in uniq if find_entity(a["caption"])[0] and a["url"] not in seen_src]print(f"kandidat: {sum(1 for a in uniq if find_entity(a['caption'])[0])} | "      f"sudah diproses: {len(seen_src)} | sisa: {len(cand)}")def prompt_core(batch):    lines = []    for i, a in enumerate(batch):        w, kat = find_entity(a["caption"])        lines.append(f"{i}. [{w} / {kat}] {a['caption']}")    return ("Ganti entitas dalam tanda kurung siku pada setiap baris. "            "Pertahankan penomoran i sesuai urutan.\n\n" + "\n".join(lines))t0 = time.time()if cand:    with open(MC_RAW, "a") as f:        for s in tqdm(range(0, len(cand), BATCH), desc="miscaptioned", unit="batch",                      disable=not VERBOSE):            batch = cand[s:s + BATCH]            try:                content, usage = call_llm([{"role": "system", "content": SYS},                                           {"role": "user", "content": prompt_core(batch)}])                got = parse_json(content)            except Exception as e:                print("batch gagal:", e)                continue            for i, a in enumerate(batch):                new = got.get(i, "")                ok, why, changed = validate(a["caption"], new)                f.write(json.dumps({"stage": "core", "source": a["url"],                                    "image": a["image"], "caption_orig": a["caption"],                                    "caption": new, "valid": ok, "reason": why,                                    "changed": changed,                                    "usage": {k: usage.get(k) for k in                                              ("prompt_tokens", "completion_tokens")},                                    "cost": (usage.get("opencode") or {}).get("list_cost")},                                   ensure_ascii=False) + "\n")            f.flush()    print(f"selesai | {time.time()-t0:.0f} detik")else:    print("semua kandidat sudah punya respons")

### 6.2 Variasi keduaSetiap artikel yang sudah punya respons inti diminta satu penggantian lain, dengan syaratberbeda dari yang pertama. Aturan validasinya sama.

In [ ]:
recs = read_jsonl(MC_RAW)base = [r for r in recs if r.get("stage") == "core" and r.get("valid")]done_var = {r["source"] for r in recs if r.get("stage") == "variant"}todo_var = [r for r in base if r["source"] not in done_var]print(f"inti: {len(base)} | variasi sudah ada: {len(done_var)} | sisa: {len(todo_var)}")SYS_VAR = (    "Kamu editor caption berita berbahasa Indonesia. Pada setiap baris ada entitas di "    "dalam tanda kurung siku, jenisnya, dan satu pengganti yang SUDAH DIPAKAI. Ganti "    "entitas itu dengan pengganti LAIN yang berbeda dari yang sudah dipakai, dari jenis "    "yang sama, sehingga caption tidak cocok dengan fotonya tetapi tetap wajar. Dilarang "    "mengubah nama orang, tempat, lembaga, tanggal, angka, dan teks dalam tanda kurung. "    "Balas HANYA dengan JSON: {\"items\":[{\"i\":0,\"caption\":\"...\"}]}.")def replacement_of(orig: str, new: str) -> str:    a, b = orig.split(), new.split()    sm = difflib.SequenceMatcher(None, a, b)    out = []    for tag, i1, i2, j1, j2 in sm.get_opcodes():        if tag != "equal":            out += b[j1:j2]    return " ".join(out)def prompt_var(batch):    lines = []    for i, r in enumerate(batch):        w, kat = find_entity(r["caption_orig"])        lines.append(f"{i}. [{w} / {kat}] sudah dipakai: "                     f"{replacement_of(r['caption_orig'], r['caption'])}. {r['caption_orig']}")    return "Ganti entitas tiap baris dengan pengganti lain.\n\n" + "\n".join(lines)t0 = time.time()if todo_var:    with open(MC_RAW, "a") as f:        for s in tqdm(range(0, len(todo_var), BATCH), desc="variasi", unit="batch",                      disable=not VERBOSE):            batch = todo_var[s:s + BATCH]            try:                content, usage = call_llm([{"role": "system", "content": SYS_VAR},                                           {"role": "user", "content": prompt_var(batch)}])                got = parse_json(content)            except Exception as e:                print("batch gagal:", e)                continue            for i, r in enumerate(batch):                new = got.get(i, "")                ok, why, changed = validate(r["caption_orig"], new)                if ok and new.strip() == r["caption"].strip():                    ok, why = False, "sama dengan variasi pertama"                f.write(json.dumps({"stage": "variant", "source": r["source"],                                    "image": r["image"], "caption_orig": r["caption_orig"],                                    "caption": new, "valid": ok, "reason": why,                                    "changed": changed,                                    "usage": {k: usage.get(k) for k in                                              ("prompt_tokens", "completion_tokens")},                                    "cost": (usage.get("opencode") or {}).get("list_cost")},                                   ensure_ascii=False) + "\n")            f.flush()    print(f"selesai | {time.time()-t0:.0f} detik")else:    print("semua variasi sudah ada")

## 7. Penyusunan dataset`miscaptioned` diambil dari respons yang lolos validasi, dengan gambar dari artikelasalnya. `true` dan `out-of-context` diambil dari langkah sebelumnya.

In [ ]:
recs = read_jsonl(MC_RAW)mc = []for r in recs:    if r.get("valid") and r.get("caption"):        mc.append({"label": "miscaptioned", "image": r["image"],                   "caption": r["caption"], "source": r["source"],                   "category": next((a["category"] for a in uniq                                     if a["url"] == r["source"]), "")})dataset = [p for p in pairs] + mcprint("total:", len(dataset), "|", Counter(p["label"] for p in dataset))print("biaya setara: $", round(sum((r.get("cost") or 0) for r in recs), 3))write_jsonl(OUT / "pairs_draft.jsonl", dataset)

## 8. Pembagian dataSplit per artikel. Artikel dengan caption identik disatukan. Pasangan `out-of-context`dibentuk ulang di dalam fold, sehingga tidak ada gambar atau caption yang menyeberang.

In [ ]:
def make_folds(rows, k=5, seed=SEED):    rng = random.Random(seed)    arts, parent, cap_owner = {}, {}, {}    def find(x):        while parent[x] != x:            parent[x] = parent[parent[x]]            x = parent[x]        return x    def union(a, b):        ra, rb = find(a), find(b)        if ra != rb:            parent[rb] = ra    for r in rows:        a = arts.setdefault(r["source"], {"n": 0, "mc": 0})        a["n"] += 1        a["mc"] += int(r["label"] == "miscaptioned")        parent.setdefault(r["source"], r["source"])        if r["label"] == "true":            cap = r["caption"].strip().lower()            if cap in cap_owner:                union(cap_owner[cap], r["source"])            else:                cap_owner[cap] = r["source"]    comp = {}    for a in arts:        comp.setdefault(find(a), []).append(a)    items = list(comp.values())    for v in items:        rng.shuffle(v)    items.sort(key=lambda v: -sum(arts[a]["mc"] for a in v))    fold_of = {}    for i, v in enumerate(items):        for a in v:            fold_of[a] = i % k    return fold_ofrows = read_jsonl(OUT / "pairs_draft.jsonl")fold_of = make_folds(rows, k=5, seed=SEED)rng_ooc = random.Random(SEED + 1)by_fold = defaultdict(list)for r in rows:    if r["label"] == "true":        by_fold[fold_of[r["source"]]].append(r)locked = []for r in tqdm(rows, desc="split", unit="pasangan", disable=not VERBOSE):    r = dict(r)    r["fold"] = fold_of[r["source"]]    if r["label"] == "out-of-context":        cand = by_fold[r["fold"]]        pool = [x for x in cand if x["image"] != r["image"]                and x.get("category") != r.get("category")]        if not pool:            pool = [x for x in cand if x["image"] != r["image"]]        r["image"] = rng_ooc.choice(pool)["image"]    locked.append(r)img_folds, leak_img = {}, 0for r in locked:    if r["image"] in img_folds and img_folds[r["image"]] != r["fold"]:        leak_img += 1    img_folds[r["image"]] = r["fold"]cap_folds, leak_cap = {}, 0for r in locked:    if r["caption"] in cap_folds and cap_folds[r["caption"]] != r["fold"]:        leak_cap += 1    cap_folds[r["caption"]] = r["fold"]print(f"pasangan: {len(locked)} | kebocoran gambar: {leak_img} | caption: {leak_cap}")write_jsonl(OUT / "pairs.jsonl", locked)

## 9. Statistik dataset

In [ ]:
import statistics as stlocked = read_jsonl(OUT / "pairs.jsonl")labels = Counter(r["label"] for r in locked)folds = Counter(r["fold"] for r in locked)panjang = [len(r["caption"].split()) for r in locked if r["label"] == "true"]kanal = Counter(r["category"] for r in locked if r["label"] == "true")kurung = sum(1 for r in locked if r["label"] == "true" and "(" in r["caption"])print("Kelas")for k, v in labels.most_common():    print(f"  {k:16s} {v:5d}  {v/len(locked)*100:5.1f}%")print("\nFold")for k in sorted(folds):    print(f"  fold {k}: {folds[k]}")print("\nCaption")print(f"  rata-rata {st.mean(panjang):.1f} kata | median {st.median(panjang)} "      f"| min {min(panjang)} | maks {max(panjang)}")print(f"  memuat kredit foto: {kurung}/{len(panjang)}")print("\nKanal teratas")for k, v in kanal.most_common(6):    print(f"  {k:16s} {v}")

## 10. Sidik jari keluaran

In [ ]:
for name in ("urls.json", "articles_raw.jsonl", "articles.jsonl",             "mc_raw.jsonl", "pairs_draft.jsonl", "pairs.jsonl"):    p = OUT / name    if p.exists():        print(f"{name:22s} {sha256_of(p)[:16]}  {p.stat().st_size/1024/1024:6.2f} MB")

## Catatan- `LIMIT` menentukan ukuran korpus. Mengubahnya mengubah komposisi dataset.- `urls.json` dan `mc_raw.jsonl` adalah titik pulih. Selama keduanya ada, menjalankan  ulang notebook tidak memanggil API lagi dan menghasilkan dataset yang sama.- Model generatif tidak deterministik. Dataset yang dilaporkan adalah hasil satu eksekusi  tertentu, dan seluruh respons mentahnya disertakan di `data/mc_raw.jsonl`.- Kunci API dibaca dari `.env`; berkas itu tidak ikut di-commit.